In [0]:
CREATE OR REPLACE PROCEDURE cf_engineering_dev.master.sp_merge_upsert(
  IN target_table_name STRING,
  IN source_view_name STRING
)
SQL SECURITY INVOKER
BEGIN
  EXECUTE IMMEDIATE '
    MERGE INTO ' || target_table_name || ' AS target
    USING ' || source_view_name || ' AS source
    ON target.car_model_code = source.car_model_code 
       AND target.process_code = source.process_code 
       AND target.design_version = source.design_version
    WHEN MATCHED AND source.sequence_number > target.sequence_number THEN
      UPDATE SET 
        target.bom_id = source.bom_id,
        target.target_cycle_time = source.target_cycle_time,
        target.sequence_number = source.sequence_number,
        target._input_file_path = source._input_file_path,
        target._processed_timestamp = source._processed_timestamp
    WHEN NOT MATCHED THEN
      INSERT (car_model_code, process_code, design_version, bom_id, target_cycle_time, sequence_number, _input_file_path, _processed_timestamp)
      VALUES (source.car_model_code, source.process_code, source.design_version, source.bom_id, source.target_cycle_time, source.sequence_number, source._input_file_path, source._processed_timestamp)
  ';
END;